In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision



import numpy as np
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import opendatasets as od
#od.download("https://www.kaggle.com/datasets/prashantsharma526/e-commerce-mens-clothing-dataset")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Dataset URL: https://www.kaggle.com/datasets/prashantsharma526/e-commerce-mens-clothing-dataset


100%|██████████| 131M/131M [00:44<00:00, 3.07MB/s] 


In [ ]:
idx2class = {
  0: 'Shirts', 1: 'Pants', 3: 'Formal Shirts', 4: 'Jeans', 5: 'Cargos', 6: 'Hoodies', 7: 'T-Shirts', 8: 'Shorts', 9: 'Solid T-Shirts'
}

In [3]:
class CNN_Model(nn.Module):
    def __init__(self, input_size, output_size):
        super(CNN_Model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(input_size, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(64 * 27 * 27, output_size)
        )

    def forward(self, x):
        x = self.seq(x)
        return x
    
model = CNN_Model(3, 8).to(device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((220, 220)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(root='./e-commerce-mens-clothing-dataset/dataset_clean', transform=transform)
dataset_test = datasets.ImageFolder(root='./e-commerce-mens-clothing-dataset/sample_images', transform=transform)
#there is not a test set -> I'll use just the train set
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False)



In [5]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


def train(model, device, train_loader, optimizer, loss_function, epochs=10):
    model.train()

    for i in range(epochs):
      total_loss = 0

      for data, target in train_loader:
          data, target = data.to(device), target.to(device)

          optimizer.zero_grad()
          output = model(data)

          loss = loss_function(output, target)
          loss.backward()
          optimizer.step()

          total_loss += loss.item() * data.size(0)
      avg_loss = total_loss / len(train_loader.dataset)
      print(f'Train Loss: {avg_loss:.4f} in epoch {i+1}/{epochs}')


In [6]:
train(model, device, train_loader, optimizer, loss_function, epochs=10)

Train Loss: 1.3038 in epoch 1/10
Train Loss: 0.8924 in epoch 2/10
Train Loss: 0.6977 in epoch 3/10
Train Loss: 0.5376 in epoch 4/10
Train Loss: 0.4103 in epoch 5/10
Train Loss: 0.2942 in epoch 6/10
Train Loss: 0.2068 in epoch 7/10
Train Loss: 0.1412 in epoch 8/10
Train Loss: 0.1077 in epoch 9/10
Train Loss: 0.0787 in epoch 10/10


In [ ]:
def evaluate(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    total_pred = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)

            loss = loss_function(output, labels)

            test_loss+= loss.item() * images.size(0)
            predicted = output.argmax(dim=1, keepdim=True)
            correct += (predicted == labels).sum().item()

            total_pred += labels.size(0)
    test_loss /= len(test_loader.dataset)
    test_acc = correct / total_pred
    return test_loss * 100, test_acc * 100

test_loss, test_acc = evaluate(model, device, train_loader)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}')
#it doesn't show proper results --> errors
  
  



Test Loss: 5.2287, Test Accuracy: 493.0551


In [1]:
#USE CASE 
image = Image.open("./image.png")
image = transform(image).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    prediction = model(image)
    print(idx2class[prediction.argmax()])

NameError: name 'Image' is not defined

Note: you may need to restart the kernel to use updated packages.
